# 04 Modeling — Calibrated Direction Models

This notebook runs the reproducible modeling pipeline in `scripts/modeling_pipeline.py`. The pipeline now uses balanced Logistic Regression, XGBoost class weighting, validation-set hyperparameter tuning, probability calibration, validation-selected classification thresholds, and balanced accuracy in the metric table.


## Source Code Note

The full reproducible modeling implementation lives in `scripts/modeling_pipeline.py`. This notebook calls that script, then loads the generated CSVs and figures so the analysis remains readable while the code stays reusable.


## 1. Run Modeling Pipeline

The script reads `daily_with_sentiment_v2.csv`, engineers sentiment lags and rolling features, trains calibrated Logit/XGB models, selects thresholds on the validation split, and writes model artifacts plus figures 1-5.


In [ ]:
from pathlib import Path
import runpy

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
FIG = ROOT / "figures"

runpy.run_path(str(ROOT / "scripts" / "modeling_pipeline.py"), run_name="__main__")


## 2. Time-Aware Split

Train is pre-2020, validation is 2020-2021, and test is 2022-2024. No random shuffle is used.


In [ ]:
display(Image(filename=str(FIG / "fig1_split_timeline.png")))


## 3. Validation-Selected Thresholds

Thresholds are selected on validation balanced accuracy and then applied unchanged to the held-out test set.


In [ ]:
thresholds = pd.read_csv(DATA / "model_thresholds.csv")
display(thresholds.round(4))


## 4. XGBoost Hyperparameter Tuning

XGBoost uses `scale_pos_weight`, and `max_depth`, `learning_rate`, and `n_estimators` are selected using validation AUC.


In [ ]:
tuning = pd.read_csv(DATA / "model_tuning_summary.csv")
best_tuning = (
    tuning.sort_values(["feature_set", "val_auc", "val_balanced_accuracy_05", "val_brier"], ascending=[True, False, False, True])
    .groupby("feature_set")
    .head(1)
    .reset_index(drop=True)
)
display(best_tuning.round(4))


## 5. Test Metrics

The main comparison includes accuracy, balanced accuracy, F1, AUC, and Brier score. Balanced accuracy is important here because plain accuracy can look acceptable even when the model mostly predicts one class.


In [ ]:
metrics = pd.read_csv(DATA / "model_metrics_summary.csv")
test_metrics = metrics[metrics["split"] == "test"].copy()
display(
    test_metrics.sort_values("balanced_accuracy", ascending=False)[
        ["model", "feature_set", "threshold", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "auc", "brier"]
    ].round(4)
)


## 6. Figures

Figures below are regenerated by the pipeline and can be used in the report/presentation.


In [ ]:
for name in [
    "fig2_sentiment_method_comparison.png",
    "fig3_logit_coefficients.png",
    "fig4_xgb_feature_importance.png",
    "fig5_roc_test.png",
]:
    display(Image(filename=str(FIG / name)))


## 7. Modeling Handoff

Main outputs for evaluation are `model_predictions.csv`, `model_metrics_summary.csv`, `model_thresholds.csv`, and `model_tuning_summary.csv`. The next notebook reads these files for regime evaluation and backtesting.


In [ ]:
preds = pd.read_csv(DATA / "model_predictions.csv", parse_dates=["Date"])
print(preds.shape)
display(preds.head())
